In [1]:
# 05_mlp_wj_distillation.ipynb
# Approach 2: fine-tune the MLP candidate generator with Weighted-Jaccard distillation.
#
# Goal:
#   Make cosine similarity in the learned 512-d embedding better imitate the
#   native WeightedJaccard teacher used by the baseline.


In [2]:
import gc
import os
import pickle
import random
import time
from pathlib import Path

import nmslib
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

THREADS = 32
QUERY_START_10K = 8000
QUERY_START_FULL = 187019

class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        return self.net(x)

class QuadtreeCompressorV1Fixed(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        x = torch.log1p(x * 1e6)
        return self.net(x)

def weighted_jaccard_np(a, b):
    mins = np.minimum(a, b).sum()
    maxs = a.sum() + b.sum() - mins
    return float(mins / max(maxs, 1e-10))

def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total = 0.0
    count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt = set(gt_lookup.get(qid, [])[:k])
        if not gt:
            continue
        total += len(gt & set(ids[:k])) / len(gt)
        count += 1
    return total / count if count else 0.0

def eval_recall(gt_lookup, nbrs, query_start_id, max_k):
    return {k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
            for k in (10, 50, 100, 500) if k <= max_k}

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2


In [3]:
# Configuration
# Start with 10k. The full dataset is possible, but pair building/training is heavier.
dataset_name = "10k"      # "10k" or "full"
device = torch.device("cuda:0")
seed = 123

positive_per_query = 5
negative_per_positive = 2
max_queries = None         # set e.g. 2000 for a faster smoke test
val_frac = 0.1

batch_size = 256
epochs = 12
lr = 1e-4
weight_decay = 1e-4
margin = 0.10
mse_weight = 1.0
rank_weight = 0.5

candidate_ks = [500, 1000]
rerank_mode = "gpu"
rerank_batch_size = 16

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [4]:
# Load cached vectors, ground truth, and the existing MLP checkpoint
if dataset_name == "10k":
    qt = np.load("/tmp/qt_10k.npy")
    with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_10K
    base_ckpt = "/tmp/best_compressor_v1_clean.pt"
    distill_ckpt = "/tmp/best_compressor_wjdistill_10k.pt"
    model_cls = QuadtreeCompressorV1
elif dataset_name == "full":
    qt = np.load("/tmp/qtree_vectors_full.npy")
    with open("/tmp/gt_lookup_full.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_FULL
    base_ckpt = "/tmp/best_compressor_full_fixed.pt"
    distill_ckpt = "/tmp/best_compressor_wjdistill_full.pt"
    model_cls = QuadtreeCompressorV1Fixed
else:
    raise ValueError(dataset_name)

corpus_qt = qt[:query_start]
query_qt = qt[query_start:]
corpus_size = len(corpus_qt)
query_ids = [qid for qid in sorted(gt) if query_start <= qid < len(qt)]
if max_queries is not None:
    query_ids = query_ids[:max_queries]

print(f"dataset={dataset_name}")
print(f"qt={qt.shape} | corpus={corpus_qt.shape} | queries={query_qt.shape}")
print(f"teacher queries used={len(query_ids)}")
print(f"base checkpoint={base_ckpt}")
print(f"distilled checkpoint={distill_ckpt}")


dataset=10k
qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)
teacher queries used=1818
base checkpoint=/tmp/best_compressor_v1_clean.pt
distilled checkpoint=/tmp/best_compressor_wjdistill_10k.pt


In [5]:
# Build weighted-Jaccard distillation triplets
# Each triplet stores: query global id, positive corpus id, negative corpus id,
# teacher WJ(q,pos), teacher WJ(q,neg).
triplets = []
for qid in tqdm(query_ids, desc="Building triplets"):
    positives = [pid for pid in gt.get(qid, []) if 0 <= pid < corpus_size]
    if not positives:
        continue
    positives = positives[:positive_per_query]
    positive_set = set(positives)
    q_vec = qt[qid]

    for pos_id in positives:
        pos_wj = weighted_jaccard_np(q_vec, corpus_qt[pos_id])
        for _ in range(negative_per_positive):
            neg_id = random.randrange(corpus_size)
            while neg_id in positive_set:
                neg_id = random.randrange(corpus_size)
            neg_wj = weighted_jaccard_np(q_vec, corpus_qt[neg_id])
            triplets.append((qid, pos_id, neg_id, pos_wj, neg_wj))

random.shuffle(triplets)
val_n = max(1, int(len(triplets) * val_frac))
val_triplets = triplets[:val_n]
train_triplets = triplets[val_n:]

print(f"triplets total={len(triplets):,} | train={len(train_triplets):,} | val={len(val_triplets):,}")
print("sample:", triplets[0] if triplets else None)


Building triplets: 100%|██████████| 1818/1818 [00:00<00:00, 1963.26it/s]

triplets total=17,440 | train=15,696 | val=1,744
sample: (9049, 5991, 3604, 0.775789737701416, 0.09842152893543243)


In [6]:
class WJDistillTripletDataset(Dataset):
    def __init__(self, qt, corpus_qt, triplets):
        self.qt = qt
        self.corpus_qt = corpus_qt
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        qid, pos_id, neg_id, pos_wj, neg_wj = self.triplets[idx]
        return (
            torch.from_numpy(self.qt[qid]).float(),
            torch.from_numpy(self.corpus_qt[pos_id]).float(),
            torch.from_numpy(self.corpus_qt[neg_id]).float(),
            torch.tensor(pos_wj, dtype=torch.float32),
            torch.tensor(neg_wj, dtype=torch.float32),
        )

train_loader = DataLoader(
    WJDistillTripletDataset(qt, corpus_qt, train_triplets),
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    WJDistillTripletDataset(qt, corpus_qt, val_triplets),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

def distill_loss(model, q, pos, neg, pos_wj, neg_wj):
    zq = F.normalize(model(q), dim=1)
    zp = F.normalize(model(pos), dim=1)
    zn = F.normalize(model(neg), dim=1)

    sim_pos = F.cosine_similarity(zq, zp)
    sim_neg = F.cosine_similarity(zq, zn)

    # Map cosine [-1, 1] to [0, 1] so it can imitate WJ teacher scores.
    pred_pos = (sim_pos + 1.0) * 0.5
    pred_neg = (sim_neg + 1.0) * 0.5
    mse = F.mse_loss(pred_pos, pos_wj) + F.mse_loss(pred_neg, neg_wj)

    rank = F.relu(margin - sim_pos + sim_neg).mean()
    loss = mse_weight * mse + rank_weight * rank
    return loss, mse.detach(), rank.detach(), sim_pos.detach().mean(), sim_neg.detach().mean()


In [7]:
# Fine-tune MLP with WJ distillation
model = model_cls(qt.shape[1], out_dim=512).to(device)
model.load_state_dict(torch.load(base_ckpt, weights_only=True, map_location=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_val = float("inf")
history = []

for epoch in range(1, epochs + 1):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs} train", leave=False)
    for q, pos, neg, pos_wj, neg_wj in pbar:
        q = q.to(device, non_blocking=True)
        pos = pos.to(device, non_blocking=True)
        neg = neg.to(device, non_blocking=True)
        pos_wj = pos_wj.to(device, non_blocking=True)
        neg_wj = neg_wj.to(device, non_blocking=True)

        loss, mse, rank, sim_pos, sim_neg = distill_loss(model, q, pos, neg, pos_wj, neg_wj)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(float(loss.detach().cpu()))
        pbar.set_postfix(loss=f"{train_losses[-1]:.4f}", mse=f"{float(mse):.4f}", rank=f"{float(rank):.4f}")

    model.eval()
    val_losses = []
    with torch.no_grad():
        for q, pos, neg, pos_wj, neg_wj in val_loader:
            q = q.to(device, non_blocking=True)
            pos = pos.to(device, non_blocking=True)
            neg = neg.to(device, non_blocking=True)
            pos_wj = pos_wj.to(device, non_blocking=True)
            neg_wj = neg_wj.to(device, non_blocking=True)
            loss, mse, rank, sim_pos, sim_neg = distill_loss(model, q, pos, neg, pos_wj, neg_wj)
            val_losses.append(float(loss.detach().cpu()))

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    scheduler.step()
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), distill_ckpt)

    print(f"Epoch {epoch:02d} | train={train_loss:.5f} | val={val_loss:.5f} | best={best_val:.5f} | lr={scheduler.get_last_lr()[0]:.2e}")

print(f"Done. Best checkpoint: {distill_ckpt}")


Epoch 01 | train=0.19807 | val=0.75281 | best=0.75281 | lr=9.83e-05


Epoch 02 | train=0.16952 | val=0.74611 | best=0.74611 | lr=9.33e-05


Epoch 03 | train=0.14560 | val=0.74553 | best=0.74553 | lr=8.54e-05


Epoch 04 | train=0.14270 | val=0.73295 | best=0.73295 | lr=7.50e-05


Epoch 05 | train=0.12234 | val=0.68895 | best=0.68895 | lr=6.29e-05


...

Epoch 08 | train=0.09552 | val=0.71327 | best=0.61370 | lr=2.50e-05


Epoch 09 | train=0.11137 | val=0.75368 | best=0.61370 | lr=1.46e-05


Epoch 10 | train=0.10273 | val=0.71199 | best=0.61370 | lr=6.70e-06


Epoch 11 | train=0.10569 | val=0.59765 | best=0.59765 | lr=1.70e-06


Epoch 12 | train=0.10480 | val=0.51817 | best=0.51817 | lr=0.00e+00
Done. Best checkpoint: /tmp/best_compressor_wjdistill_10k.pt


In [8]:
# Quick embedding quality check: does distillation improve GT-vs-random cosine gap?
def embedding_quality(model, qt, gt, query_start, corpus_size, n=500):
    model.eval()
    qids = [qid for qid in sorted(gt) if query_start <= qid < len(qt)]
    qids = qids[:n]
    gt_sims, rand_sims = [], []
    with torch.no_grad():
        for qid in tqdm(qids, desc="Quality check"):
            pos_id = gt.get(qid, [None])[0]
            if pos_id is None or pos_id >= corpus_size:
                continue
            rand_id = random.randrange(corpus_size)
            q = torch.tensor(qt[qid], dtype=torch.float32, device=device).unsqueeze(0)
            p = torch.tensor(qt[pos_id], dtype=torch.float32, device=device).unsqueeze(0)
            r = torch.tensor(qt[rand_id], dtype=torch.float32, device=device).unsqueeze(0)
            eq = F.normalize(model(q), dim=1)
            ep = F.normalize(model(p), dim=1)
            er = F.normalize(model(r), dim=1)
            gt_sims.append(F.cosine_similarity(eq, ep).item())
            rand_sims.append(F.cosine_similarity(eq, er).item())
    return float(np.mean(gt_sims)), float(np.mean(rand_sims)), float(np.mean(gt_sims) - np.mean(rand_sims))

base_model = model_cls(qt.shape[1], out_dim=512).to(device)
base_model.load_state_dict(torch.load(base_ckpt, weights_only=True, map_location=device))

distilled_model = model_cls(qt.shape[1], out_dim=512).to(device)
distilled_model.load_state_dict(torch.load(distill_ckpt, weights_only=True, map_location=device))

base_quality = embedding_quality(base_model, qt, gt, query_start, corpus_size)
distill_quality = embedding_quality(distilled_model, qt, gt, query_start, corpus_size)

print(f"Base      GT={base_quality[0]:.4f} | Rand={base_quality[1]:.4f} | Gap={base_quality[2]:.4f}")
print(f"Distilled GT={distill_quality[0]:.4f} | Rand={distill_quality[1]:.4f} | Gap={distill_quality[2]:.4f}")


Quality check: 100%|██████████| 500/500 [00:00<00:00, 787.86it/s]

Base      GT=0.9923 | Rand=0.7966 | Gap=0.1957
Distilled GT=0.9883 | Rand=0.6712 | Gap=0.3171


In [10]:
# Optional: evaluate distilled checkpoint with the two-stage GPU reranker.
# This duplicates the important pieces so the notebook is self-contained.
def generate_embeddings(model, data, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[start:start + batch_size], dtype=torch.float32, device=device)
            chunks.append(F.normalize(model(batch), dim=1).cpu().numpy())
    return np.vstack(chunks)

def build_cosine_index(corpus_embs):
    m0 = get_mem_mb()
    idx = nmslib.init(method="hnsw", space="cosinesimil")
    for i in tqdm(range(len(corpus_embs)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({"M": 20, "efConstruction": 200, "post": 1}, print_progress=True)
    build_s = time.time() - t0
    idx_mb = get_mem_mb() - m0
    idx.setQueryTimeParams({"efSearch": 200})
    return idx, build_s, idx_mb

def rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    reranked = [None] * len(nbrs_raw)
    for start in tqdm(range(0, len(nbrs_raw), batch_size), desc="GPU WJ rerank"):
        batch = nbrs_raw[start:start + batch_size]
        groups = {}
        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for _, items in groups.items():
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[absolute_i] for absolute_i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=device)
            q_t = torch.from_numpy(query_np).to(device=device, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order = torch.argsort(mins / maxs.clamp_min(1e-10), dim=1, descending=True).cpu().numpy()
            for row, (absolute_i, ids) in zip(order, items):
                reranked[absolute_i] = (ids[row].tolist(), [])
    del corpus_t, corpus_sums_t
    torch.cuda.empty_cache()
    return reranked

def evaluate_two_stage(model, label):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    embs = generate_embeddings(model, qt, device)
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    vec_mb = corpus_embs.nbytes / 1024**2
    idx, build_s, idx_mb = build_cosine_index(corpus_embs)
    corpus_sums = corpus_qt.sum(axis=1)
    results = {}
    for k in candidate_ks:
        t0 = time.time()
        nbrs_raw = idx.knnQueryBatch(query_embs, k=k, num_threads=THREADS)
        hnsw_s = time.time() - t0
        t0 = time.time()
        nbrs_rr = rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, rerank_batch_size)
        rerank_s = time.time() - t0
        qps = len(query_embs) / (hnsw_s + rerank_s)
        rec = eval_recall(gt, nbrs_rr, query_start, max_k=k)
        results[f"k{k}_wj_rerank"] = {**rec, "qps": qps, "hnsw_s": hnsw_s, "rerank_s": rerank_s,
                                      "build_s": build_s, "vec_mb": vec_mb, "idx_mb": idx_mb}
        print(f"K={k} | QPS={qps:.1f} | HNSW={hnsw_s:.2f}s | WJ={rerank_s:.2f}s")
        for kk, rr in rec.items():
            print(f"  R@{kk:<4} = {rr:.4f}")
    return results

# Uncomment when you are ready to run the full retrieval evaluation.
distill_eval_results = evaluate_two_stage(distilled_model, "Distilled MLP + cosine candidates + GPU WJ rerank")



Distilled MLP + cosine candidates + GPU WJ rerank


Adding: 100%|██████████| 8000/8000 [00:00<00:00, 674908.62it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 263.27it/s]


K=500 | QPS=3419.2 | HNSW=0.07s | WJ=0.51s
  R@10   = 0.9912
  R@50   = 0.9770
  R@100  = 0.9518
  R@500  = 0.8553


GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 133.61it/s]


K=1000 | QPS=1850.0 | HNSW=0.10s | WJ=0.98s
  R@10   = 0.9949
  R@50   = 0.9869
  R@100  = 0.9686
  R@500  = 0.8857
